In [43]:
import re
import pickle

import numpy as np
import pandas as pd
# import dask.dataframe as pd
import sqlite3

from sklearn.preprocessing import StandardScaler

#import xgboost

con = sqlite3.connect('../Dengue20X_timeseries_CPA_NoiseReduction.db')

cursor = con.cursor()
cursor.execute('SELECT name FROM sqlite_master WHERE type="table";')
print(cursor.fetchall())

[('MyExpt_Per_Object',), ('MyExpt_Per_Image',), ('Experiment',), ('sqlite_sequence',), ('Experiment_Properties',), ('MyExpt_Per_Experiment',), ('MyExpt_Per_RelationshipTypes',), ('MyExpt_Per_Relationships',), ('NS4B_Epro_scored_imagelevel',), ('NS4B_scored_imagelevel',), ('Epro_scored_imagelevel',), ('NS4B_Epro_objectlevelUMAP',), ('UMAP_sample20',), ('_link_tables_MyExpt_Per_Image_MyExpt_Per_Object_',), ('_link_columns_MyExpt_Per_Image_MyExpt_Per_Object_',), ('UMAP_sample10',), ('NS4B_Epro_objectUMAP_infectedONLY',), ('NS4B_Epro_objectUMAP_sample10Redo',), ('NS4B_Epro_objectlevelUMAP_redo',)]


In [3]:
conn = sqlite3.connect('Z:\Active_Users_Data\Jillian\DENV_assay_dvlp\CPOutput_DR\Dengue20X_timeseries_CPA_DR.db')

In [15]:
meta = pd.read_sql_query('SELECT ImageNumber, Image_Metadata_WellID, Image_Metadata_PlateID from MyExpt_Per_Image', con)
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [16]:
meta_DR = pd.read_sql_query('SELECT ImageNumber, Image_Metadata_WellID, Image_Metadata_PlateID from MyExpt_Per_Image', conn)
meta_DR.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,DR_20221209_143805
1,2,A01,DR_20221209_143805
2,3,A01,DR_20221209_143805
3,4,A01,DR_20221209_143805
4,5,A01,DR_20221209_143805


In [6]:
def _letter_range(start, stop="{", step=1):
    """Yield a range of lowercase letters.""" 
    for ord_ in range(ord(start.upper()), ord(stop.upper()), step):
        yield chr(ord_)

PC, NC = [],[]
for l in list(_letter_range('A', 'Q')):
    for i in range(1, 5):
        NC.append(f"{l}{str(i).zfill(2)}")
    for j in range(5, 9):
        PC.append(f"{l}{str(j).zfill(2)}")

print(NC)
print()
print(PC)

['A01', 'A02', 'A03', 'A04', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G02', 'G03', 'G04', 'H01', 'H02', 'H03', 'H04', 'I01', 'I02', 'I03', 'I04', 'J01', 'J02', 'J03', 'J04', 'K01', 'K02', 'K03', 'K04', 'L01', 'L02', 'L03', 'L04', 'M01', 'M02', 'M03', 'M04', 'N01', 'N02', 'N03', 'N04', 'O01', 'O02', 'O03', 'O04', 'P01', 'P02', 'P03', 'P04']

['A05', 'A06', 'A07', 'A08', 'B05', 'B06', 'B07', 'B08', 'C05', 'C06', 'C07', 'C08', 'D05', 'D06', 'D07', 'D08', 'E05', 'E06', 'E07', 'E08', 'F05', 'F06', 'F07', 'F08', 'G05', 'G06', 'G07', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'J08', 'K05', 'K06', 'K07', 'K08', 'L05', 'L06', 'L07', 'L08', 'M05', 'M06', 'M07', 'M08', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O07', 'O08', 'P05', 'P06', 'P07', 'P08']


In [7]:
def _letter_range(start, stop="{", step=1):
    """Yield a range of lowercase letters.""" 
    for ord_ in range(ord(start.upper()), ord(stop.upper()), step):
        yield chr(ord_)

PC_DR, NC_DR = [],[]
for l in list(_letter_range('A', 'Q')):
    for i in range(1, 3):
        PC_DR.append(f"{l}{str(i).zfill(2)}")
    for j in range(23, 25):
        NC_DR.append(f"{l}{str(j).zfill(2)}")


In [8]:
highCount = pd.read_csv('WellswHighCellCount.csv')
highCountList = highCount['Image_Metadata_WellID'].unique().tolist()
highCountList

['A01',
 'A02',
 'A03',
 'B01',
 'B02',
 'B03',
 'B04',
 'C01',
 'C02',
 'C03',
 'C04',
 'D01',
 'D02',
 'D03',
 'D04',
 'E01',
 'E02',
 'E03',
 'E04',
 'F01',
 'F02',
 'F03',
 'F04',
 'G01',
 'G04',
 'H01',
 'H03',
 'H04',
 'I04',
 'J01',
 'J02',
 'K01',
 'L03',
 'L04',
 'M01',
 'M02',
 'M04',
 'N01',
 'N02',
 'N04',
 'O01',
 'O04',
 'P01',
 'P03',
 'P04',
 'A21',
 'A22',
 'A23',
 'A24',
 'B21',
 'B22',
 'B23',
 'C22',
 'C23',
 'D21',
 'D22',
 'D23',
 'D24',
 'E21',
 'E22',
 'E23',
 'E24',
 'F21',
 'F23',
 'G21',
 'G22',
 'G23',
 'G24',
 'H21',
 'H22',
 'H23',
 'I21',
 'I22',
 'I23',
 'I24',
 'J21',
 'J22',
 'J23',
 'J24',
 'K21',
 'K22',
 'K23',
 'K24',
 'L21',
 'L22',
 'L23',
 'L24',
 'M21',
 'M22',
 'M23',
 'M24',
 'N21',
 'N22',
 'N23',
 'N24',
 'O21',
 'O22',
 'O23',
 'A17',
 'A19',
 'B17',
 'B19',
 'C17',
 'G17',
 'G19',
 'H20',
 'I17',
 'I18',
 'I19',
 'I20',
 'J18',
 'K17',
 'K18',
 'K20',
 'L18',
 'L19',
 'M17',
 'M20',
 'N17',
 'O18',
 'O19',
 'O20',
 'P17',
 'P20',
 'A14',


In [11]:
lowCount_DR = ["N01"]

In [9]:
#select only PC and NC with high cell count to train
PC_highCC = set(PC).intersection(set(highCountList))
NC_highCC = set(NC).intersection(set(highCountList))
print(PC_highCC)
print()
print(NC_highCC)

{'J06', 'A05', 'G08', 'D05', 'G06', 'E06', 'O08', 'I06', 'E05', 'B07', 'F05', 'F06', 'L07', 'O05', 'C05', 'L06', 'P05', 'J07', 'D06', 'L05', 'E08', 'H07', 'C07', 'A06', 'B05', 'A08', 'I05', 'C06', 'N06', 'N05', 'H05', 'N08', 'O06', 'P06', 'J05', 'K05', 'G05', 'I08', 'M05', 'M07', 'B06', 'I07', 'H06', 'N07', 'H08'}

{'F01', 'M02', 'M01', 'D02', 'H03', 'J02', 'O01', 'E04', 'B03', 'J01', 'B02', 'C03', 'B01', 'L03', 'H04', 'E03', 'C01', 'D01', 'F04', 'A01', 'N02', 'I04', 'G01', 'P03', 'D04', 'H01', 'C04', 'G04', 'D03', 'M04', 'C02', 'A03', 'F02', 'B04', 'L04', 'P01', 'K01', 'F03', 'A02', 'N04', 'E02', 'O04', 'P04', 'E01', 'N01'}


In [13]:
PC_highCC_DR = set(PC_DR) - set(lowCount_DR)
NC_highCC_DR = set(NC_DR)

print(PC_highCC_DR)
print(NC_highCC_DR)

{'F01', 'M02', 'M01', 'D02', 'J02', 'O01', 'L02', 'J01', 'B02', 'I01', 'I02', 'B01', 'C01', 'D01', 'A01', 'N02', 'G01', 'O02', 'H01', 'C02', 'F02', 'P01', 'P02', 'K01', 'A02', 'H02', 'L01', 'E02', 'K02', 'E01', 'G02'}
{'J24', 'E24', 'K24', 'B23', 'F23', 'D23', 'E23', 'G24', 'O23', 'M23', 'N24', 'O24', 'D24', 'M24', 'H24', 'B24', 'L23', 'J23', 'I23', 'H23', 'A24', 'C23', 'L24', 'A23', 'P23', 'K23', 'I24', 'N23', 'P24', 'G23', 'C24', 'F24'}


In [10]:
PC = list(PC_highCC)
PC.sort()
print(PC)
NC = list(NC_highCC)
NC.sort()
print(NC)

['A05', 'A06', 'A08', 'B05', 'B06', 'B07', 'C05', 'C06', 'C07', 'D05', 'D06', 'E05', 'E06', 'E08', 'F05', 'F06', 'G05', 'G06', 'G08', 'H05', 'H06', 'H07', 'H08', 'I05', 'I06', 'I07', 'I08', 'J05', 'J06', 'J07', 'K05', 'L05', 'L06', 'L07', 'M05', 'M07', 'N05', 'N06', 'N07', 'N08', 'O05', 'O06', 'O08', 'P05', 'P06']
['A01', 'A02', 'A03', 'B01', 'B02', 'B03', 'B04', 'C01', 'C02', 'C03', 'C04', 'D01', 'D02', 'D03', 'D04', 'E01', 'E02', 'E03', 'E04', 'F01', 'F02', 'F03', 'F04', 'G01', 'G04', 'H01', 'H03', 'H04', 'I04', 'J01', 'J02', 'K01', 'L03', 'L04', 'M01', 'M02', 'M04', 'N01', 'N02', 'N04', 'O01', 'O04', 'P01', 'P03', 'P04']


In [77]:
PC_DR = list(PC_highCC_DR)
PC_DR.sort()
print(PC_DR)
NC_DR = list(NC_highCC_DR)
NC_DR.sort()
print(NC_DR)

['A01', 'A02', 'B01', 'B02', 'C01', 'C02', 'D01', 'D02', 'E01', 'E02', 'F01', 'F02', 'G01', 'G02', 'H01', 'H02', 'I01', 'I02', 'J01', 'J02', 'K01', 'K02', 'L01', 'L02', 'M01', 'M02', 'N02', 'O01', 'O02', 'P01', 'P02']
['A23', 'A24', 'B23', 'B24', 'C23', 'C24', 'D23', 'D24', 'E23', 'E24', 'F23', 'F24', 'G23', 'G24', 'H23', 'H24', 'I23', 'I24', 'J23', 'J24', 'K23', 'K24', 'L23', 'L24', 'M23', 'M24', 'N23', 'N24', 'O23', 'O24', 'P23', 'P24']


In [17]:
meta = meta.loc[meta['Image_Metadata_WellID'].isin(NC+PC)]
wells = meta['ImageNumber'].unique().tolist()
meta.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,TimeSeries_20221028_164132
1,2,A01,TimeSeries_20221028_164132
2,3,A01,TimeSeries_20221028_164132
3,4,A01,TimeSeries_20221028_164132
4,5,A01,TimeSeries_20221028_164132


In [20]:
meta_DR = meta_DR.loc[meta_DR['Image_Metadata_WellID'].isin(NC_DR+PC_DR)]
wells_DR  = meta_DR['ImageNumber'].unique().tolist()
meta_DR.head()

,ImageNumber,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,A01,DR_20221209_143805
1,2,A01,DR_20221209_143805
2,3,A01,DR_20221209_143805
3,4,A01,DR_20221209_143805
4,5,A01,DR_20221209_143805


In [22]:
query ="SELECT * FROM MyExpt_Per_Object WHERE ImageNumber IN("

In [23]:
query+=f"{wells[0]}"
for w in wells[1:]:
    query+=","
    query+=f"{w}"
query+=")"

In [24]:
data = pd.read_sql_query(query, con)
data.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,0.0,0.0,9.712645,10.273859,9.904125,10.098906,0.0,0.0,0.0,0.0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,0.0,0.0,6.579749,6.315129,5.832482,6.946683,0.0,0.0,0.0,0.0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,0.0,0.0,15.497243,16.163633,15.251661,16.716883,0.0,0.0,0.0,0.0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,0.0,0.0,13.691379,13.019686,13.500400,13.272403,0.0,0.0,0.0,0.0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.0,0.0,0.673830,0.660640,0.689110,0.654166,0.0,0.0,0.0,0.0


In [25]:
query ="SELECT * FROM MyExpt_Per_Object WHERE ImageNumber IN("

In [26]:
query+=f"{wells_DR[0]}"
for w in wells_DR[1:]:
    query+=","
    query+=f"{w}"
query+=")"

In [27]:
data_DR = pd.read_sql_query(query, conn)
data_DR.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_Children_Cytoplasm_Count,Cells_Intensity_IntegratedIntensityEdge_CMO,Cells_Intensity_IntegratedIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensityEdge_Hoe,Cells_Intensity_IntegratedIntensityEdge_NS4BAfterMath,Cells_Intensity_IntegratedIntensity_CMO,Cells_Intensity_IntegratedIntensity_EproAfterMath,...,Nuclei_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_Variance_Hoe_6_00_256,Nuclei_Texture_Variance_Hoe_6_01_256,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256
0,1,1,1,1,23.907286,0.519448,2.775036,4.973587,522.031205,7.976730,...,0.072288,0.057448,28.295443,27.638773,28.751241,28.428445,24.547365,26.646684,24.652204,23.749182
1,1,2,2,1,15.694881,0.449882,2.222629,0.598016,1154.960877,34.370032,...,2.763491,2.831337,382.755946,381.204094,382.230064,416.673265,203.171620,197.638112,219.224613,230.024473
2,1,3,3,1,19.664775,1.322606,0.618189,2.934310,599.849913,53.021043,...,6.005855,6.303360,102.632628,103.548271,104.205093,105.998847,5.135475,4.789403,4.604633,5.250396
3,1,4,4,1,22.014008,0.561547,2.242557,2.601038,577.120852,7.746365,...,0.090000,0.000000,44.092351,44.706518,44.925159,47.016393,13.561260,14.813755,18.665295,27.369501
4,1,5,5,1,20.394720,0.859388,1.041077,1.407919,969.678707,43.190479,...,1.826888,1.298090,46.517867,46.624956,45.973323,48.713327,2.029531,2.244061,1.963110,1.597432


In [53]:
df = pd.merge(data, meta, on='ImageNumber')
df['Image_Metadata_WellID'].unique()

array(['A01', 'A02', 'A03', 'A05', 'A06', 'A08', 'B01', 'B02', 'B03',
       'B04', 'B05', 'B06', 'B07', 'C01', 'C02', 'C03', 'C04', 'C05',
       'C06', 'C07', 'D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'E01',
       'E02', 'E03', 'E04', 'E05', 'E06', 'E08', 'F01', 'F02', 'F03',
       'F04', 'F05', 'F06', 'G01', 'G04', 'G05', 'G06', 'G08', 'H01',
       'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'I04', 'I05', 'I06',
       'I07', 'I08', 'J01', 'J02', 'J05', 'J06', 'J07', 'K01', 'K05',
       'L03', 'L04', 'L05', 'L06', 'L07', 'M01', 'M02', 'M04', 'M05',
       'M07', 'N01', 'N02', 'N04', 'N05', 'N06', 'N07', 'N08', 'O01',
       'O04', 'O05', 'O06', 'O08', 'P01', 'P03', 'P04', 'P05', 'P06'],
      dtype=object)

In [30]:
df_DR = pd.merge(data_DR, meta_DR, on='ImageNumber')
df_DR['Image_Metadata_WellID'].unique()

array(['A01', 'A02', 'A23', 'A24', 'B01', 'B02', 'B23', 'B24', 'C01',
       'C02', 'C23', 'C24', 'D01', 'D02', 'D23', 'D24', 'E01', 'E02',
       'E23', 'E24', 'F01', 'F02', 'F23', 'F24', 'G01', 'G02', 'G23',
       'G24', 'H01', 'H02', 'H23', 'H24', 'I01', 'I02', 'I23', 'I24',
       'J01', 'J02', 'J23', 'J24', 'K01', 'K02', 'K23', 'K24', 'L01',
       'L02', 'L23', 'L24', 'M01', 'M02', 'N02', 'O01', 'O02'],
      dtype=object)

In [54]:
df['con'] = 'NC'
df.loc[df['Image_Metadata_WellID'].isin(PC), 'con'] = 'PC'

In [32]:
df_DR['con'] = 'NC'
df_DR.loc[df_DR['Image_Metadata_WellID'].isin(PC_DR), 'con'] = 'PC'

In [55]:
df['label'] = 0
df.loc[df['con']=='PC', 'label'] = 1

In [34]:
df_DR['label'] = 0
df_DR.loc[df_DR['con']=='PC', 'label'] = 1

In [56]:
df.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_AreaShape_Area,Cells_AreaShape_BoundingBoxArea,Cells_AreaShape_BoundingBoxMaximum_X,Cells_AreaShape_BoundingBoxMaximum_Y,Cells_AreaShape_BoundingBoxMinimum_X,Cells_AreaShape_BoundingBoxMinimum_Y,Cells_AreaShape_Center_X,...,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID,con,label
0,1,1,1,2565.0,4080.0,622.0,48.0,537.0,0.0,576.887329,...,9.904125,10.098906,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
1,1,2,2,1682.0,2090.0,1179.0,38.0,1124.0,0.0,1154.139120,...,5.832482,6.946683,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
2,1,3,3,3226.0,5900.0,1548.0,59.0,1448.0,0.0,1496.832300,...,15.251661,16.716883,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
3,1,4,4,6035.0,8804.0,745.0,71.0,621.0,0.0,676.520133,...,13.500400,13.272403,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0
4,1,5,5,1527.0,2244.0,1219.0,51.0,1175.0,0.0,1193.948265,...,0.689110,0.654166,0.0,0.0,0.0,0.0,A01,TimeSeries_20221028_164132,NC,0


In [36]:
df_DR.head()

,ImageNumber,ObjectNumber,Cells_Number_Object_Number,Cells_Children_Cytoplasm_Count,Cells_Intensity_IntegratedIntensityEdge_CMO,Cells_Intensity_IntegratedIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensityEdge_Hoe,Cells_Intensity_IntegratedIntensityEdge_NS4BAfterMath,Cells_Intensity_IntegratedIntensity_CMO,Cells_Intensity_IntegratedIntensity_EproAfterMath,...,Nuclei_Texture_Variance_Hoe_6_02_256,Nuclei_Texture_Variance_Hoe_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Image_Metadata_WellID,Image_Metadata_PlateID,con,label
0,1,1,1,1,23.907286,0.519448,2.775036,4.973587,522.031205,7.976730,...,28.751241,28.428445,24.547365,26.646684,24.652204,23.749182,A01,DR_20221209_143805,PC,1
1,1,2,2,1,15.694881,0.449882,2.222629,0.598016,1154.960877,34.370032,...,382.230064,416.673265,203.171620,197.638112,219.224613,230.024473,A01,DR_20221209_143805,PC,1
2,1,3,3,1,19.664775,1.322606,0.618189,2.934310,599.849913,53.021043,...,104.205093,105.998847,5.135475,4.789403,4.604633,5.250396,A01,DR_20221209_143805,PC,1
3,1,4,4,1,22.014008,0.561547,2.242557,2.601038,577.120852,7.746365,...,44.925159,47.016393,13.561260,14.813755,18.665295,27.369501,A01,DR_20221209_143805,PC,1
4,1,5,5,1,20.394720,0.859388,1.041077,1.407919,969.678707,43.190479,...,45.973323,48.713327,2.029531,2.244061,1.963110,1.597432,A01,DR_20221209_143805,PC,1


In [37]:
meta_cols = df.columns[df.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [40]:
meta_cols_DR = df_DR.columns[df_DR.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [38]:
cols = df.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [41]:
cols_DR = df_DR.drop(columns=meta_cols_DR).select_dtypes(include='float64').columns.tolist()

In [44]:
#FOR DR
with open('data_cols_reduced_DR', 'rb') as f:
    common_data_cols = pickle.load(f)

In [45]:
common_data_cols

['Nuclei_Intensity_MassDisplacement_NS4BAfterMath',
 'Nuclei_Texture_Contrast_EproAfterMath_6_03_256',
 'Cells_Intensity_MaxIntensityEdge_EproAfterMath',
 'Cells_Intensity_IntegratedIntensity_NS4BAfterMath',
 'Cells_Intensity_MedianIntensity_EproAfterMath',
 'Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256',
 'Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256',
 'Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256',
 'Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_00_256',
 'Nuclei_Texture_Entropy_NS4BAfterMath_6_03_256',
 'Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_03_256',
 'Nuclei_Intensity_UpperQuartileIntensity_EproAfterMath',
 'Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_00_256',
 'Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_01_256',
 'Nuclei_Texture_InfoMeas1_EproAfterMath_6_00_256',
 'Cells_Intensity_UpperQuartileIntensity_EproAfterMath',
 'Nuclei_Texture_InfoMeas2_EproAfterMath_6_03_256',
 'Nuclei_Texture_Correlation_EproAfterMath_6_00_256',
 'Nuc

In [46]:
print(df.shape)
print(df_DR.shape)

(663795, 978)
(144059, 690)


In [51]:
DR_cols = df_DR.columns.tolist()

In [58]:
df = df.loc[:, DR_cols]

In [59]:
df.shape

(663795, 690)

In [60]:
concat_dfs = pd.concat([df, df_DR])

In [61]:
concat_dfs.shape

(807854, 690)

In [63]:
concat_dfs[common_data_cols] = StandardScaler().fit_transform(concat_dfs[common_data_cols])

In [64]:
import xgboost
model = xgboost.XGBRegressor()

In [65]:
from sklearn.model_selection import train_test_split

In [66]:
X = concat_dfs[common_data_cols]
y = concat_dfs['label']

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [68]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=100, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)

In [27]:
import pickle

In [28]:
with open('data_cols_reduced', 'wb') as f:
    pickle.dump(data_cols, f)

In [69]:
preds = model.predict(X_test)

In [70]:
from sklearn.metrics import r2_score, mean_squared_error

r2 = r2_score(y_true=y_test, y_pred=preds)
mse = mean_squared_error(y_true=y_test, y_pred=preds)

In [71]:
print("R2", r2)
print("MSE", mse)

R2 0.9020176446104561
MSE 0.024204564655359393


In [72]:
model.save_model('xgb_model_0_48_wDRcontrols')

In [73]:
del df

In [74]:
del df_DR

In [75]:
del concat_dfs